In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from pathlib import Path

id_attrs = [
    "variable_id",
    "domain_id",
    "driving_source_id",
    "driving_experiment_id",
    "driving_variant_label",
    "institution_id",
    "source_id",
    "version_realization",
    "frequency",
    "version",
]


def filename_to_id(filename):
    """
    Extract the dataset id from the filename.
    """
    stem = Path(filename).stem
    path = str(Path(filename).parent)
    version = path.split("/")[-1]
    values = stem.split("_")[0 : len(id_attrs) - 1] + [version]
    return ".".join(values)


def filename_to_attrs(filename):
    """
    Create a dictionary with the dataset id as key and the filename as value.
    """
    stem = Path(filename).stem
    path = str(Path(filename).parent)
    version = path.split("/")[-1]
    values = stem.split("_")[0 : len(id_attrs) - 1] + [version]
    return dict(zip(id_attrs, values))


with open("report/compliance-report.json") as fp:
    cc_data = json.load(fp)

In [ ]:
import pandas as pd

prios = {
    "cf": ["low_priorities", "medium_priorities", "high_priorities"],
    "cc6": ["low_priorities", "medium_priorities", "high_priorities"],
}

cols = ["scored_points", "possible_points", "high_count", "medium_count", "low_count"]

id_attrs = [
    "variable_id",
    "domain_id",
    "driving_source_id",
    "driving_experiment_id",
    "driving_variant_label",
    "institution_id",
    "source_id",
    "version_realization",
    "frequency",
    "version",
]


def concat_messages(tests):
    summary = ""
    for test in tests:
        if test.get("msgs"):
            summary += "\n".join(test["msgs"]) + "\n"
    return summary


def summarize(test, results):
    summaries = {}
    test_id = test.split(":")[0]
    summaries = {f"{test_id}:{c}": results[c] for c in cols}
    for prio in prios[test_id]:
        tests = results.get(prio, [])
        summary = concat_messages(tests)
        summaries[f"{test_id}:{prio}"] = summary
    return summaries


def to_dataframe(cc_data):
    df = (
        pd.DataFrame.from_dict(cc_data, orient="index")
        .reset_index()
        .rename(columns={"index": "filename"})
    )
    return df


result = {}

for filename, tests in cc_data.items():
    summary = {}
    for test, results in tests.items():
        summary.update(summarize(test, results))
    result[filename] = filename_to_attrs(filename) | summary

In [5]:
corrupt = pd.read_csv("report/corrupt_files.csv")
df = (
    pd.DataFrame.from_dict(result, orient="index")
    .reset_index()
    .rename(columns={"index": "filename"})
)
df.to_csv("report/compliance-report.csv", index=False)
df = df.merge(corrupt, on="filename", how="left")

In [7]:
df

,filename,variable_id,domain_id,driving_source_id,driving_experiment_id,driving_variant_label,institution_id,source_id,version_realization,frequency,...,cc6:high_priorities,cf:scored_points,cf:possible_points,cf:high_count,cf:medium_count,cf:low_count,cf:low_priorities,cf:medium_priorities,cf:high_priorities,not_readable
0,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,hus600,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute hus600:_QuantizeBitGroomNumberOfSign...,,NaN
1,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,hus700,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute hus700:_QuantizeBitGroomNumberOfSign...,,NaN
2,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,hus850,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute hus850:_QuantizeBitGroomNumberOfSign...,,NaN
3,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,hus925,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute hus925:_QuantizeBitGroomNumberOfSign...,,NaN
4,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,ta850,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,392,0,2,0,,attribute ta850:_QuantizeBitGroomNumberOfSigni...,,NaN
5,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,ua1000,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute ua1000:_QuantizeBitGroomNumberOfSign...,,NaN
6,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,ua200,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute ua200:_QuantizeBitGroomNumberOfSigni...,,NaN
7,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,ua250,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute ua250:_QuantizeBitGroomNumberOfSigni...,,NaN
8,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,ua300,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute ua300:_QuantizeBitGroomNumberOfSigni...,,NaN
9,/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD...,ua400,EUR-12,NorESM2-MM,historical,r1i1p1f1,ICTP,RegCM5-0,v1-r1,6hr,...,DRS path building block 'frequency' does not c...,390,391,0,1,0,,attribute ua400:_QuantizeBitGroomNumberOfSigni...,,NaN


In [ ]:
import os


def human_readable(df):
    """
    Creates a human-readable summary of the dataset.

    Parameters:
    df (pandas.DataFrame): The input DataFrame containing the dataset.

    Returns:
    pandas.DataFrame: A DataFrame with grouped and summarized data.
    """

    index = [
        "institution_id",
        "domain_id",
        "source_id",
        "driving_experiment_id",
        "driving_source_id",
        "driving_variant_label",
        "version_realization",
        "variable_id",
        "frequency",
        "version",
        "filename",
    ]
    # cols = [c for c in df.columns if c not in index]
    return df.sort_values(index).set_index(index)


def create_excel(filename):
    """
    Creates a human-readable Excel file from the dataset.

    Parameters:
    filename (str): The path to the CSV file containing the dataset.

    Returns:
    str: The path to the created Excel file.
    """
    df = pd.read_csv(filename)
    sheets = {"jsc-cordex": human_readable(df)}

    stem, suffix = os.path.splitext(filename)
    xlsxfile = f"{stem}.xlsx"

    with pd.ExcelWriter(xlsxfile, engine="xlsxwriter") as writer:
        for sheet_name, sheet_df in sheets.items():
            print(sheet_name)
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=True)
            worksheet = writer.sheets[sheet_name]  # pull worksheet object

            # Set the column width to the maximum width of the content
            for idx, col in enumerate(sheet_df.columns):
                max_len = (
                    max(
                        sheet_df[col].astype(str).map(len).max(),  # len of largest item
                        len(str(col)),  # len of column name/header
                    )
                    + 2
                )  # adding a little extra space
                worksheet.set_column(
                    idx + len(sheet_df.index.names),
                    idx + len(sheet_df.index.names),
                    max_len,
                )

            # Set the column width for the index levels
            for idx, level in enumerate(sheet_df.index.names):
                max_len = (
                    max(
                        sheet_df.index.get_level_values(level)
                        .astype(str)
                        .map(len)
                        .max(),  # len of largest item in index level
                        len(str(level)),  # len of index level name
                    )
                    + 2
                )  # adding a little extra space
                worksheet.set_column(idx, idx, max_len)

    return xlsxfile

In [10]:
create_excel("report/compliance-report.csv")

jsc-cordex


'report/compliance-report.xlsx'

In [ ]:
df = pd.read_csv("report/compliance-report.csv")
human_readable(df).to_excel("report/compliance-report.xlsx", index=True)

In [ ]:
df